In [1]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from lazypredict.Supervised import LazyRegressor

In [2]:
# reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
# models, predictions = reg.fit(X_train, X_test, y_train, y_test)

# print(models)

In [3]:
df = pd.read_csv('5.2-)CAR DETAILS FROM CAR DEKHO.csv')

In [4]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           4340 non-null   object
 1   year           4340 non-null   int64 
 2   selling_price  4340 non-null   int64 
 3   km_driven      4340 non-null   int64 
 4   fuel           4340 non-null   object
 5   seller_type    4340 non-null   object
 6   transmission   4340 non-null   object
 7   owner          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB


In [6]:
df['fuel'].unique()

array(['Petrol', 'Diesel', 'CNG', 'LPG', 'Electric'], dtype=object)

In [7]:
df['seller_type'].unique()

array(['Individual', 'Dealer', 'Trustmark Dealer'], dtype=object)

In [8]:
df['transmission'].value_counts()

transmission
Manual       3892
Automatic     448
Name: count, dtype: int64

In [9]:
df['owner'].unique()

array(['First Owner', 'Second Owner', 'Fourth & Above Owner',
       'Third Owner', 'Test Drive Car'], dtype=object)

In [10]:
df['transmission'] = np.where(df['transmission'] == 'Manual' , 1 , 0) # Manual = 1 

In [11]:
df = pd.get_dummies(df , columns = ['seller_type'] , drop_first = True)

In [12]:
df.head()

,name,year,selling_price,km_driven,fuel,transmission,owner,seller_type_Individual,seller_type_Trustmark Dealer
0,Maruti 800 AC,2007,60000,70000,Petrol,1,First Owner,True,False
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,1,First Owner,True,False
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,1,First Owner,True,False
3,Datsun RediGO T Option,2017,250000,46000,Petrol,1,First Owner,True,False
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,1,Second Owner,True,False


In [13]:
df['fuel'] = df['fuel'].map({
    'Petrol' : 0 ,
    'Diesel' : 1 , 
    'CNG'    : 2 , 
    'LPG'    : 3 ,
    'Electric' : 4 
})

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
encoder = LabelEncoder()

In [16]:
df['owner'] = encoder.fit_transform(df['owner'])

In [17]:
df.head()

,name,year,selling_price,km_driven,fuel,transmission,owner,seller_type_Individual,seller_type_Trustmark Dealer
0,Maruti 800 AC,2007,60000,70000,0,1,0,True,False
1,Maruti Wagon R LXI Minor,2007,135000,50000,0,1,0,True,False
2,Hyundai Verna 1.6 SX,2012,600000,100000,1,1,0,True,False
3,Datsun RediGO T Option,2017,250000,46000,0,1,0,True,False
4,Honda Amaze VX i-DTEC,2014,450000,141000,1,1,2,True,False


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   name                          4340 non-null   object
 1   year                          4340 non-null   int64 
 2   selling_price                 4340 non-null   int64 
 3   km_driven                     4340 non-null   int64 
 4   fuel                          4340 non-null   int64 
 5   transmission                  4340 non-null   int64 
 6   owner                         4340 non-null   int64 
 7   seller_type_Individual        4340 non-null   bool  
 8   seller_type_Trustmark Dealer  4340 non-null   bool  
dtypes: bool(2), int64(6), object(1)
memory usage: 245.9+ KB


In [19]:
df['seller_type_Individual'] = df['seller_type_Individual'].astype(int)

In [20]:
df['seller_type_Trustmark Dealer'] = df['seller_type_Trustmark Dealer'].astype(int)

In [21]:
# Sadece ilk kelimeyi (markayı) al
df['brand'] = df['name'].apply(lambda x: x.split(' ')[0])

# Artık 'name' sütununu silebiliriz
df.drop(columns=['name'], inplace=True)

# Kategorik encoding
df = pd.get_dummies(df, columns=['brand'], drop_first=True)


In [22]:
def correlation_for_dropping(df , treshold) :
    corr = df.corr()
    columns_to_drop = set()
    for i in range(len(corr.columns)) : 
        for j in range(i) : 
            if abs(corr.iloc[i , j]) > treshold : 
                columns_to_drop.add(corr.columns[i])
    return columns_to_drop

In [23]:
# -------------- MACHINE LEARNING ---------------

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
X = df.drop(columns = ['selling_price'] , axis = 1)
y = df['selling_price']

In [26]:
X_train , X_test , y_train ,y_test = train_test_split(X , y , random_state = 50 , test_size = 0.25)

In [27]:
columns_dropped = correlation_for_dropping(X_train , 0.83)
columns_dropped

set()

In [28]:
X_train.drop(columns = columns_dropped , inplace = True, axis = 1)
X_test.drop(columns = columns_dropped , inplace = True , axis = 1)

In [29]:
X_train = X_train.drop(columns = ['brand_Force'])
X_test = X_test.drop(columns = ['brand_Force'])

In [30]:
from sklearn.preprocessing import StandardScaler

In [31]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.linear_model import LinearRegression

regression = LinearRegression()
regression.fit(X_train , y_train)

y_pred = regression.predict(X_test)

from sklearn.metrics import r2_score

r2 = r2_score(y_test , y_pred)
print(r2)


0.6479701653120015


In [32]:
 reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
 models, predictions = reg.fit(X_train, X_test, y_train, y_test)

 print(models)

  0%|          | 0/42 [00:00<?, ?it/s]

  File "C:\Users\MSI-NB\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\MSI-NB\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MSI-NB\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MSI-NB\anaconda3\Lib\subproc

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002923 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 346
[LightGBM] [Info] Number of data points in the train set: 3255, number of used features: 24
[LightGBM] [Info] Start training from score 511847.397849
                               Adjusted R-Squared  R-Squared        RMSE  \
Model                                                                      
XGBRegressor                                 0.82       0.82   227949.78   
ExtraTreesRegressor                          0.82       0.82   229062.89   
RandomForestRegressor                        0.81       0.82   231341.87   
GradientBoostingRegressor                    0.80       0.80   239630.67   
BaggingRegressor                             0.79       0.79   246606.87   
KNeighborsRegressor                          0.78       0.7